# 03 — Patient-Level Split

**Goal of this notebook**: split `metadata.csv` into train/val/test such that every
slice from one patient lands in exactly one split. This is the direct fix for the
leakage issue found in the commonly used Kaggle version of this dataset (slice-level
split let the same patient's slices appear in both train and test).

**Intent of each step below**: apply the split, then *prove* it's leakage-free with an
explicit assertion — not just trust the function did the right thing.

In [ ]:
import sys

sys.path.insert(0, "..")

import pandas as pd

from src.data_utils import SplitConfig, assert_no_patient_leakage, patient_level_split

metadata = pd.read_csv("../data/processed/metadata.csv")

## Apply the split

`patient_level_split` internally calls `assert_no_patient_leakage` already — but we
verify it again explicitly below as documentation, not just trust.

In [ ]:
split_metadata = patient_level_split(metadata, SplitConfig(test_size=0.2, val_size=0.1, seed=42))
split_metadata["split"].value_counts()

## Explicit leakage proof

**Intent**: this is the check that the leaky Kaggle dataset never had. Print it clearly
so it's documented evidence in the notebook output, not just a passing assertion.

In [ ]:
assert_no_patient_leakage(split_metadata)

overlap_check = split_metadata.groupby("patient_id")["split"].nunique()
assert (overlap_check == 1).all()
print(
    f"Confirmed: all {split_metadata['patient_id'].nunique()} unique patient/group IDs "
    f"appear in exactly one split."
)

## Class balance per split

**Intent**: patient-level grouping can distort class proportions across splits more than
a naive random split would (since we're moving whole patients, not individual images).
Check it's still reasonably balanced.

In [ ]:
pd.crosstab(split_metadata["label"], split_metadata["split"], normalize="columns")

In [ ]:
split_metadata.to_csv("../data/processed/metadata_split.csv", index=False)
print("Saved: data/processed/metadata_split.csv")

Next notebook: **04_baseline_custom_cnn.ipynb** — train the from-scratch CNN
baseline on this clean split.